# ChunkLab Notebook: Text Chunking for RAG
**Time:** about 45 minutes · **Needs:** Python 3.9+, no installs (standard library only)

By the end you will be able to:
1. Build fixed-size, sentence-aware and recursive chunkers from scratch
2. Explain how size and overlap change what retrieval finds
3. Rank chunks with TF-IDF and measure retrieval with a hit rate
4. Attach metadata so answers can cite their source

Pace: Parts 1-2 (12 min), Part 3 (8 min), Exercises 1-3 (20 min), wrap-up (5 min). Work top to bottom, run every cell, and try each exercise before opening the solutions.

In [ ]:
import re, math
from collections import Counter

DOC = """Refund Policy

Customers may request a refund within 30 days of purchase. Refunds are issued to the original payment method.

Digital products are non-refundable once downloaded. Shipping fees are not refunded unless the item arrived damaged.

To start a request, contact support at help@shop.com with your order number."""
print(len(DOC), "characters, about", len(DOC)//4, "tokens")

## Part 1: Fixed-size chunking
Cut every `size` characters, stepping forward by `size - overlap`. Fast, but blind to meaning.

In [ ]:
def fixed_chunks(text, size, overlap=0):
    step = max(1, size - overlap)
    out = []
    for i in range(0, len(text), step):
        out.append(text[i:i+size])
        if i + size >= len(text):
            break
    return out

def show(chunks):
    for i, c in enumerate(chunks, 1):
        print(f"[{i}] ({len(c)} chars) {c!r}")

show(fixed_chunks(DOC, 60))

**Look closely:** where does "30 days" end up? Run it with `size=60`, then change the size to `55`. A fact cut in half cannot be retrieved whole.

## Part 2: Sentence-aware and recursive chunking
Pack whole sentences up to `size`. If `overlap > 0`, the next chunk starts with the previous chunk's last sentence. The recursive version keeps paragraphs intact when they fit, and only falls back to sentences when they don't.

In [ ]:
def sentences(t):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", t) if s]

def pack(units, size, overlap):
    out, cur = [], []
    for u in units:
        if cur and len(" ".join(cur)) + len(u) + 1 > size:
            out.append(" ".join(cur))
            cur = [cur[-1]] if overlap else []
        cur.append(u)
    if cur:
        out.append(" ".join(cur))
    return out

def sentence_chunks(text, size, overlap=0):
    return pack(sentences(text), size, overlap)

def recursive_chunks(text, size, overlap=0):
    out = []
    for p in re.split(r"\n\n+", text):
        out += [p] if len(p) <= size else pack(sentences(p), size, overlap)
    return out

for name, fn in [("fixed", fixed_chunks), ("sentence", sentence_chunks), ("recursive", recursive_chunks)]:
    print("\n==", name)
    show(fn(DOC, 160))

## Part 3: Retrieval from scratch (TF-IDF)
Real systems use embeddings, but TF-IDF shows the same idea: score every chunk against the question and take the best. Words that appear in few chunks count more.

In [ ]:
STOP = set("how many do i have to a the of is are what can my in on for and or be it when".split())

def tokens(t):
    return [w.rstrip("s") for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOP]

def rank(query, chunks):
    docs = [Counter(tokens(c)) for c in chunks]
    n = len(chunks)
    df = Counter(w for d in docs for w in d)
    idf = {w: math.log((n + 1) / (df[w] + 1)) + 1 for w in df}
    vec = lambda c: {w: c[w] * idf.get(w, 1) for w in c}
    q = vec(Counter(tokens(query)))
    def cos(a, b):
        dot = sum(a[w] * b.get(w, 0) for w in a)
        na = math.sqrt(sum(v * v for v in a.values()))
        nb = math.sqrt(sum(v * v for v in b.values()))
        return dot / (na * nb) if na and nb else 0.0
    scores = [cos(q, vec(d)) for d in docs]
    order = sorted(range(n), key=lambda i: -scores[i])
    return order, scores

question = "How many days do I have to request a refund?"
chunks = fixed_chunks(DOC, 60)
order, scores = rank(question, chunks)
print("Top chunk:", repr(chunks[order[0]]))
print("Contains the full fact?", "30 days" in chunks[order[0]])

**Token note:** characters / 4 is a rough estimate for English. Production systems count real tokens (for example with the `tiktoken` library) because model and embedding limits are in tokens.

## Exercise 1: make retrieval work (mirrors the website challenge)

In [ ]:
STRATEGY = "recursive"
SIZE = 160
OVERLAP = 0

In [ ]:
fn = {"fixed": fixed_chunks, "sentence": sentence_chunks, "recursive": recursive_chunks}[STRATEGY]
chunks = fn(DOC, SIZE, OVERLAP)
order, _ = rank(question, chunks)
top = chunks[order[0]]
print("Top chunk:", repr(top))
assert SIZE <= 160, "Size must be 160 or less"
assert "30 days" in top, "Top chunk does not contain the full '30 days' fact"
assert top.strip()[-1] in ".!?", "Top chunk ends mid-sentence"
print("Exercise 1 passed")

## Exercise 2: measure retrieval with a hit rate

In [ ]:
QA = [
    ("How many days do I have to request a refund?", "30 days"),
    ("Can I get a refund on digital products?", "non-refundable"),
    ("When are shipping fees refunded?", "arrived damaged"),
    ("How do I start a refund request?", "help@shop.com"),
]

def hit_rate(chunker, size, overlap=0, k=1):
    chunks = chunker(DOC, size, overlap)
    hits = 0
    for q, expected in QA:
        order, _ = rank(q, chunks)
        if any(expected in chunks[i] for i in order[:k]):
            hits += 1
    return hits / len(QA)

In [ ]:
for name, fn in [("fixed", fixed_chunks), ("sentence", sentence_chunks), ("recursive", recursive_chunks)]:
    print(f"{name:10s} size=60: {hit_rate(fn, 60):.2f}   size=160: {hit_rate(fn, 160):.2f}")
assert 0 <= hit_rate(recursive_chunks, 160) <= 1
assert hit_rate(recursive_chunks, 160) >= hit_rate(fixed_chunks, 60), "Recursive/160 should not lose to fixed/60"
print("Exercise 2 passed. Which strategy wins, and why?")

## Exercise 3: keep metadata with every chunk

In [ ]:
def add_metadata(chunks, source, text):
    return [{"id": i, "source": source, "start": text.find(c), "text": c} for i, c in enumerate(chunks)]

In [ ]:
records = add_metadata(recursive_chunks(DOC, 160), "refund_policy.txt", DOC)
for r in records: print(r["id"], r["source"], r["start"], r["text"][:40])
assert all({"id", "source", "start", "text"} <= set(r) for r in records)
assert records[1]["start"] > records[0]["start"]
print("Exercise 3 passed. This is what lets answers cite where they came from.")

## Wrap-up
- Chunk boundaries decide whether a fact survives. Measure with real questions (hit rate), not by eye.
- Match the strategy to the document: Q&A pairs for FAQs, functions for code, speaker turns for transcripts.
- Next steps: replace `rank` with embeddings (for example `sentence-transformers`), store vectors in a vector database, and evaluate with 20 real user questions.

**Reflection:** change the question in Part 3 to something your own document would be asked. Does your chunking still hold up?